In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets
!pip install peft
!pip install evaluate scikit-learn transformers[torch]
!pip install accelerate>=0.26.0

Looking in indexes: https://download.pytorch.org/whl/cu121


# Prepare the dataset

In [2]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = imdb.map(tokenize_function, batched=True)


/tmp/mdiop_tp5/env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   4%|▍         | 1000/25000 [00:00<00:04, 5208.37 examples/s]

Map:   8%|▊         | 2000/25000 [00:00<00:04, 5160.66 examples/s]

Map:  12%|█▏        | 3000/25000 [00:00<00:04, 5135.09 examples/s]

...

Map:  88%|████████▊ | 22000/25000 [00:04<00:00, 5088.93 examples/s]

Map:  92%|█████████▏| 23000/25000 [00:04<00:00, 5073.75 examples/s]

Map:  96%|█████████▌| 24000/25000 [00:04<00:00, 5091.40 examples/s]

Map: 100%|██████████| 25000/25000 [00:04<00:00, 5103.11 examples/s]

Map: 100%|██████████| 25000/25000 [00:04<00:00, 5012.46 examples/s]

In [3]:
training_set = tokenized_datasets['train'].shuffle(42).select(range(5000))
shuffled_set =  tokenized_datasets['test'].shuffle(42)
validation_set  = shuffled_set.select(range(1000))
test_set = shuffled_set.select(range(1000, 2000))

## First method : Using the transformers library and pytorch


In [4]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification



## Fine-tuning the whole model

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-cased', num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [7]:
import copy
from transformers import TrainingArguments, Trainer

def count_trainable_parameters(model):
    count = 0
    for parameter in model.parameters():
        if parameter.requires_grad:
            count = count + parameter.numel()
    return count

# we fine-tune a copy, so that model stays the pretrained BERT used for LoRA below
full_model = copy.deepcopy(model)

# same training arguments as for LoRA below, to compare the approaches fairly
training_args = TrainingArguments(output_dir="test_full_finetuning",
                                  eval_strategy="steps",
                                  eval_steps= 128,
                                  num_train_epochs=2,)
full_trainer = Trainer(
    model=full_model,
    args=training_args,
    train_dataset= training_set,
    eval_dataset=validation_set,
    compute_metrics=compute_metrics,
)
full_train_output = full_trainer.train()
full_test_results = full_trainer.evaluate(test_set)
print("Trainable parameters:", count_trainable_parameters(full_model))
print("Test accuracy:", full_test_results["eval_accuracy"])

Step,Training Loss,Validation Loss,Accuracy
128,No log,0.631433,0.823000
256,No log,0.388977,0.862000
384,No log,0.333574,0.877000
512,0.430800,0.317666,0.871000
640,0.430800,0.359763,0.883000
768,0.430800,0.470236,0.884000
896,0.430800,0.371156,0.896000
1024,0.246100,0.400577,0.893000
1152,0.246100,0.428654,0.896000


Trainable parameters: 108311810
Test accuracy: 0.909


## LoRA implementation

In [8]:
[k for k,v  in model.bert.named_parameters()]

['embeddings.word_embeddings.weight',
 'embeddings.position_embeddings.weight',
 'embeddings.token_type_embeddings.weight',
 'embeddings.LayerNorm.weight',
 'embeddings.LayerNorm.bias',
 'encoder.layer.0.attention.self.query.weight',
 'encoder.layer.0.attention.self.query.bias',
 'encoder.layer.0.attention.self.key.weight',
 'encoder.layer.0.attention.self.key.bias',
 'encoder.layer.0.attention.self.value.weight',
 'encoder.layer.0.attention.self.value.bias',
 'encoder.layer.0.attention.output.dense.weight',
 'encoder.layer.0.attention.output.dense.bias',
 'encoder.layer.0.attention.output.LayerNorm.weight',
 'encoder.layer.0.attention.output.LayerNorm.bias',
 'encoder.layer.0.intermediate.dense.weight',
 'encoder.layer.0.intermediate.dense.bias',
 'encoder.layer.0.output.dense.weight',
 'encoder.layer.0.output.dense.bias',
 'encoder.layer.0.output.LayerNorm.weight',
 'encoder.layer.0.output.LayerNorm.bias',
 'encoder.layer.1.attention.self.query.weight',
 'encoder.layer.1.attention.se

In [9]:
print(f"What is the model \n\n {model}")


What is the model 

 BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Lay

### Create the LoRA module

We will consider in this module the original linear, and the down and up projection

In [10]:
import torch 
import copy
from torch import nn

class LoRALinear(nn.Module):
  def __init__(
    self, in_dim: int, out_dim: int, rank: int
  ):
    super().__init__()
    self.linear = nn.Linear(in_dim, out_dim)
    # Q = W x + B (A x), with A of size rank x in_dim and B of size out_dim x rank
    self.lora_a = nn.Linear(in_dim, rank, bias=False)
    self.lora_b = nn.Linear(rank, out_dim, bias=False)
    # B starts at zero as in the LoRA paper: at the beginning the output is exactly the pretrained one
    nn.init.zeros_(self.lora_b.weight)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    frozen_out = self.linear(x)
    lora_out = self.lora_b(self.lora_a(x))
    return frozen_out + lora_out

### Create a function replacing linear by lora module
From the original Linear module return a LoRA module
* The linear of the LoRA linear must be initialised with pretrained weights

In [11]:


def linear_to_lora(linear):
    linear_weight = linear.weight.data
    has_bias = linear.bias is not None
    if has_bias:
        linear_bias = linear.bias.data
    output_size, input_size =  linear_weight.shape
    lora = LoRALinear(input_size, output_size, rank=8)
    lora.linear.weight.data = linear_weight
    if has_bias:
        lora.linear.bias.data = linear_bias
    return lora
    


We now replace the target linear by the LoRALinear described above

In [12]:
lora_model = copy.deepcopy(model)
lora_parameters = []
for block in lora_model.bert.encoder.layer:
    # as in the course, the key, value and query projections of each self-attention are adapted
    block.attention.self.key = linear_to_lora(block.attention.self.key)
    block.attention.self.value = linear_to_lora(block.attention.self.value)
    block.attention.self.query = linear_to_lora(block.attention.self.query)

In [13]:
print(f"What is the model modified with LoRA: \n\n {lora_model}")


What is the model modified with LoRA: 

 BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): LoRALinear(
                (linear): Linear(in_features=768, out_features=768, bias=True)
                (lora_a): Linear(in_features=768, out_features=8, bias=False)
                (lora_b): Linear(in_features=8, out_features=768, bias=False)
              )
              (key): LoRALinear(
                (linear): Linear(in_features=768, out_features=768, bias=True)
                (lora

### Defining module requiring grad

In [14]:
for k,v  in lora_model.bert.named_parameters():
    print(k)
    if ('lora' in k):
        v.requires_grad = True
    else:
        v.requires_grad = False

embeddings.word_embeddings.weight
embeddings.position_embeddings.weight
embeddings.token_type_embeddings.weight
embeddings.LayerNorm.weight
embeddings.LayerNorm.bias
encoder.layer.0.attention.self.query.linear.weight
encoder.layer.0.attention.self.query.linear.bias
encoder.layer.0.attention.self.query.lora_a.weight
encoder.layer.0.attention.self.query.lora_b.weight
encoder.layer.0.attention.self.key.linear.weight
encoder.layer.0.attention.self.key.linear.bias
encoder.layer.0.attention.self.key.lora_a.weight
encoder.layer.0.attention.self.key.lora_b.weight
encoder.layer.0.attention.self.value.linear.weight
encoder.layer.0.attention.self.value.linear.bias
encoder.layer.0.attention.self.value.lora_a.weight
encoder.layer.0.attention.self.value.lora_b.weight
encoder.layer.0.attention.output.dense.weight
encoder.layer.0.attention.output.dense.bias
encoder.layer.0.attention.output.LayerNorm.weight
encoder.layer.0.attention.output.LayerNorm.bias
encoder.layer.0.intermediate.dense.weight
encode

## Using transformers Trainer to fine-tune with LoRA 

In [15]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="test_custom_lora", 
                                  eval_strategy="steps",
                                  eval_steps= 128,
                                  num_train_epochs=2,)
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset= training_set,
    eval_dataset=validation_set,
    compute_metrics=compute_metrics,
)


In [16]:
lora_train_output = trainer.train()
lora_test_results = trainer.evaluate(test_set)
# the classifier is outside lora_model.bert, so it is still trained (it is new and has to be learned)
print("Trainable parameters (LoRA + classifier):", count_trainable_parameters(lora_model))
print("Test accuracy:", lora_test_results["eval_accuracy"])

Step,Training Loss,Validation Loss,Accuracy
128,No log,0.662235,0.611000
256,No log,0.628804,0.675000
384,No log,0.536799,0.733000
512,0.624100,0.413558,0.823000
640,0.624100,0.367162,0.841000
768,0.624100,0.357590,0.849000
896,0.624100,0.335325,0.862000
1024,0.370800,0.338146,0.860000
1152,0.370800,0.331592,0.864000


Trainable parameters (LoRA + classifier): 443906
Test accuracy: 0.879


## Using PEFT library to fine-tune with LoRA 

In [17]:
from peft import LoraConfig, TaskType,  get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=1, lora_alpha=1, lora_dropout=0.1
)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-cased', 
    num_labels=2
)
peft_model = get_peft_model(model, lora_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
peft_model.print_trainable_parameters()

trainable params: 38,402 || all params: 108,350,212 || trainable%: 0.0354


In [19]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="test_custom_lora", 
                                  eval_strategy="steps",
                                  eval_steps= 128,
                                  num_train_epochs=2,)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset= training_set,
    eval_dataset=validation_set,
    compute_metrics=compute_metrics,
)

In [20]:
peft_train_output = trainer.train()
peft_test_results = trainer.evaluate(test_set)
print("Test accuracy:", peft_test_results["eval_accuracy"])

Step,Training Loss,Validation Loss,Accuracy
128,No log,0.679633,0.563000
256,No log,0.673519,0.599000
384,No log,0.666454,0.626000
512,0.688500,0.652393,0.650000
640,0.688500,0.647103,0.647000
768,0.688500,0.623497,0.673000
896,0.688500,0.612784,0.676000
1024,0.651800,0.602514,0.684000
1152,0.651800,0.596565,0.693000


Test accuracy: 0.699


## Compare the results of the different approaches

In [21]:
results = [
    ("Full fine-tuning", count_trainable_parameters(full_model), full_test_results["eval_accuracy"], full_train_output.metrics["train_runtime"]),
    ("Our LoRA (r=8)", count_trainable_parameters(lora_model), lora_test_results["eval_accuracy"], lora_train_output.metrics["train_runtime"]),
    ("PEFT LoRA (r=1)", count_trainable_parameters(peft_model), peft_test_results["eval_accuracy"], peft_train_output.metrics["train_runtime"]),
]
print(f"{'Approach':<20}{'Trainable parameters':>22}{'Test accuracy':>15}{'Training time (s)':>19}")
for name, parameters, accuracy, runtime in results:
    print(f"{name:<20}{parameters:>22,}{accuracy:>15.3f}{runtime:>19.0f}")

Approach              Trainable parameters  Test accuracy  Training time (s)
Full fine-tuning               108,311,810          0.909                554
Our LoRA (r=8)                     443,906          0.879                464
PEFT LoRA (r=1)                     38,402          0.699                460


## Description of our choices

We use the same training arguments for the three approaches (2 epochs on the 5000 training reviews, with the default learning rate and batch size of the Trainer), so that only the adaptation method changes. The accuracy is computed on the 1000 test reviews, which are not the 1000 validation reviews used during training. We moved the cell defining the metric before the full fine-tuning, because it needs it.

The full fine-tuning is done on a copy of the model, so that the LoRA model starts from the pretrained BERT and not from the fine-tuned one.

In our LoRA module, the original linear layer keeps its pretrained weights and is frozen, and we add two matrices without bias: A (rank x input size) and B (output size x rank), so the output is Wx + B(Ax) as in the course. B is initialised at zero as in the LoRA paper, so at the beginning BA = 0 and the model is exactly the pretrained BERT. Like in the course, we replace the key, value and query projections of the 12 layers, with a rank of 8. The trained parameters are the LoRA matrices (12 x 3 x 8 x (768 + 768) = 442,368) and the classifier (1,538), which is outside lora_model.bert and has to be learned since it is new, so 443,906 in total.

For PEFT we kept the configuration given in the notebook (r = 1, alpha = 1). By default PEFT only adapts the query and value projections of BERT, which gives 12 x 2 x 1 x (768 + 768) + 1,538 = 38,402 trained parameters.

## Conclusion

The full fine-tuning gives the best accuracy (0.909), but it trains the 108 million parameters of BERT. Our LoRA with a rank of 8 reaches 0.879 while training only 443,906 parameters (0.4% of the model), and its validation accuracy was still slowly increasing at the end of the 2 epochs. The PEFT LoRA with a rank of 1 trains even fewer parameters (38,402) but only reaches 0.699: with a rank of 1 on two projections and the default learning rate, it does not adapt enough to the task in 2 epochs.

The training time is only slightly shorter with LoRA (464 s against 554 s), because the forward and backward passes still go through the whole network. As seen in the course, the gain of LoRA is mainly in memory: the optimizer only keeps the states of the LoRA parameters, and for a new task only A and B need to be saved. At inference, BA can be added to W, so the adapted model is not slower than the original one. So LoRA is a good choice when the model is too large to be fully fine-tuned, as long as the rank is not too small.